<a href="https://colab.research.google.com/github/Susan-sapkota-98/Nepali-asr-evaluation/blob/main/05_run_model_on_200.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 14.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jiwer]


In [3]:
import csv
from pathlib import Path

import jiwer
import pandas as pd
import soundfile as sf
import torch
from google.colab import drive
from transformers import Wav2Vec2ForCTC, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("using:", device)

using: cuda


In [5]:
drive.mount("/content/drive")

save_folder = Path("/content/drive/MyDrive/nepali-asr-evaluation/rating_200")
save_folder.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [6]:
work_folder = Path("/content/openslr54")
work_folder.mkdir(exist_ok=True)

base_url = "https://openslr.trmal.net/resources/54"
!wget -q -c -P {work_folder} {base_url}/utt_spk_text.tsv
!wget -q -c -P {work_folder} {base_url}/asr_nepali_0.zip
!unzip -q -n {work_folder}/asr_nepali_0.zip -d {work_folder}/audio_zip_0

In [7]:
all_rows = pd.read_csv(
    work_folder / "utt_spk_text.tsv",
    sep="\t",
    header=None,
    names=["file_id", "speaker_id", "text"],
    dtype=str,
    keep_default_na=False,
    quoting=csv.QUOTE_NONE,
)

audio_files = list((work_folder / "audio_zip_0").rglob("*.flac"))
path_by_id = {}
for file_path in audio_files:
    path_by_id[file_path.stem] = file_path

all_rows["audio_path"] = all_rows["file_id"].map(path_by_id)
rows_with_audio = all_rows.dropna(subset=["audio_path"])
print(len(rows_with_audio), "rows with audio")

9935 rows with audio


In [8]:
chosen_rows = rows_with_audio.sample(n=200, random_state=7).reset_index(drop=True)
chosen_rows["audio_path"] = chosen_rows["audio_path"].astype(str)
print(len(chosen_rows), "files chosen")

200 files chosen


In [9]:
model_name = "facebook/mms-1b-all"
processor = AutoProcessor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name)

processor.tokenizer.set_target_lang("npi")
model.load_adapter("npi")
model = model.to(device)
model.eval()
print("model ready")

preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.04k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.86GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

adapter.npi.safetensors: reconstructing file:   0%|          |  0.00B / 9.28MB            

adapter.npi.safetensors: downloading bytes:           |  0.00B            

model ready


In [10]:
def get_prediction(audio_file):
    audio, sample_rate = sf.read(audio_file)
    model_input = processor(audio, sampling_rate=sample_rate, return_tensors="pt").to(device)
    with torch.no_grad():
        scores = model(**model_input).logits
    best_ids = torch.argmax(scores, dim=-1)[0]
    return processor.decode(best_ids)

In [11]:
model_texts = []
for number, audio_path in enumerate(chosen_rows["audio_path"]):
    model_texts.append(get_prediction(audio_path))
    if (number + 1) % 50 == 0:
        print(number + 1, "done")

chosen_rows["model_text"] = model_texts

results = chosen_rows[["file_id", "text", "model_text"]].rename(columns={"text": "correct_text"})
results.to_csv(save_folder / "predictions_200.csv", index=False)

50 done
100 done
150 done
200 done


In [12]:
results["wer"] = [jiwer.wer(c, m) for c, m in zip(results["correct_text"], results["model_text"])]
results["cer"] = [jiwer.cer(c, m) for c, m in zip(results["correct_text"], results["model_text"])]

print("WER:", round(jiwer.wer(list(results["correct_text"]), list(results["model_text"])) * 100, 1), "%")
print("CER:", round(jiwer.cer(list(results["correct_text"]), list(results["model_text"])) * 100, 1), "%")

results.to_csv(save_folder / "scores_200.csv", index=False)

WER: 58.2 %
CER: 18.6 %


In [13]:
different_rows = results[results["correct_text"] != results["model_text"]]

rating_sheet = different_rows[["file_id", "correct_text", "model_text"]].copy()
rating_sheet["rater_1"] = ""
rating_sheet["rater_2"] = ""
rating_sheet["rater_3"] = ""

rating_sheet.to_csv(save_folder / "rating_sheet.csv", index=False)
print(len(rating_sheet), "rows to rate, out of", len(results))
rating_sheet.head()

145 rows to rate, out of 200


,file_id,correct_text,model_text,rater_1,rater_2,rater_3
0,0be7aaf743,वुड्सको प्रदर्शन प्रभावशाली,हुटस्कव प्रदर्शन प्रभाव शाली,,,
1,0a85921e16,मन्दिरमा तीनवटा हिन्दू,मन्दिरमा तिनोटा हिन्दु,,,
4,0e8e0445a1,सङ्घर्षका कार्यक्रमहरू पनि,संगर्षका कार्यक्रमहरू पनि,,,
5,0acd8c6b40,उहाँ धौलागिरी अञ्चलबाट,उहाँ धलाग्री अञ्चलबाट,,,
7,0e98526682,पहिलो लक्ष्य हुने,पहिलो लच्ष्य हुने,,,
